# Kaggle inference: final_direction_model

Runs the final BERT direction model from Hugging Face repo `wairado/post_direction_notator` on `RELEVANCE.parquet`. The notebook builds topic-post pairs from existing relevance/topic predictions as `[TOPIC=t_i] text`, predicts `+` / `-`, and saves the final post-level table to `preds.parquet`.


### Шаг 1. Настройка окружения и путей

Здесь задаются точный Kaggle-путь к входному parquet, локальный fallback `RELEVANCE.parquet`, Hugging Face repo модели `wairado/post_direction_notator`, режим выбора тем `predicted` и единственный выходной файл `preds.parquet`.


In [ ]:
from pathlib import Path
import gc
import json
import os
import time
import warnings

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working') if IS_KAGGLE else Path('data/kaggle_working/direction')
WORKING.mkdir(parents=True, exist_ok=True)

KAGGLE_INPUT_PARQUET = Path('/kaggle/input/datasets/wairado/spermatoksikoz-archivecore/.parquet')
INPUT_FILE_NAME = 'RELEVANCE.parquet'
TOPIC_PRED_FILE_NAME = None
HF_MODEL_ID = 'wairado/post_direction_notator'
HF_REVISION = 'main'
HF_CACHE_DIR = WORKING / 'hf_cache'
OUTPUT_FILE_NAME = 'preds.parquet'
TEXT_COL = 'text'
ID_COL = 'post_uid'
DEFAULT_TOPICS = ['t1', 't2', 't3', 't4', 't5']

# predicted | gold | all | auto
PAIR_TOPIC_SOURCE = 'predicted'
RUN_ALL_TOPICS_IF_NO_TOPIC_OUTPUT = True

BATCH_SIZE_GPU = 8
BATCH_SIZE_CPU = 2
MAX_ROWS = None       # e.g. 100 for a fast smoke run
LIMIT_PAIRS = None    # e.g. 200 for debugging inference
SAVE_CSV = False

print('IS_KAGGLE:', IS_KAGGLE)
print('WORKING:', WORKING.resolve())


### Шаг 2. Проверка зависимостей

Ячейка проверяет библиотеки для BERT-инференса, чтения parquet и расчёта метрик. Если в Kaggle runtime чего-то не хватает, пакет ставится автоматически перед запуском модели.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow',
    'torch': 'torch',
    'transformers': 'transformers',
    'huggingface_hub': 'huggingface_hub',
    'safetensors': 'safetensors',
    'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
}
missing = [pip_name for import_name, pip_name in required.items() if importlib.util.find_spec(import_name) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())


### Шаг 3. Поиск входа и настройка Hugging Face модели

Ноутбук сначала проверяет точный Kaggle-путь `/kaggle/input/datasets/wairado/spermatoksikoz-archivecore/.parquet`, затем ищет `RELEVANCE.parquet` как fallback. Direction-модель берётся из Hugging Face repo `wairado/post_direction_notator`; cache складывается в рабочую директорию. В Kaggle для этого должен быть включён Internet.


In [ ]:
def unique_existing(paths):
    seen = set()
    out = []
    for p in paths:
        p = Path(p)
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def find_file(file_name, local_fallbacks=(), required=True, prefer_working=True):
    candidates = []
    if prefer_working:
        candidates.append(WORKING / file_name)
    candidates.extend(Path(p) for p in local_fallbacks)
    candidates.append(Path.cwd() / file_name)
    if IS_KAGGLE:
        candidates.extend(KAGGLE_INPUT.glob(f'**/{file_name}'))
    candidates.extend(Path.cwd().glob(f'**/{file_name}'))

    for p in unique_existing(candidates):
        if p.exists() and p.is_file():
            return p.resolve()
    if required:
        raise FileNotFoundError(f'Could not find {file_name}. Add it as a Kaggle dataset or put it in the project tree.')
    return None


if KAGGLE_INPUT_PARQUET.exists():
    input_path = KAGGLE_INPUT_PARQUET.resolve()
else:
    input_path = find_file(INPUT_FILE_NAME, ['data/RELEVANCE.parquet'])
topic_pred_path = find_file(TOPIC_PRED_FILE_NAME, required=False) if TOPIC_PRED_FILE_NAME else None
model_id = HF_MODEL_ID
model_revision = HF_REVISION
hf_cache_dir = HF_CACHE_DIR
hf_cache_dir.mkdir(parents=True, exist_ok=True)

print('INPUT:', input_path)
print('TOPIC_PREDICTIONS:', topic_pred_path)
print('HF_MODEL_ID:', model_id)
print('HF_REVISION:', model_revision)
print('HF_CACHE_DIR:', hf_cache_dir.resolve())


### Шаг 4. Чтение конфигурации модели

Из `direction_model_config.json` и `config.json`, скачанных из Hugging Face repo, читаются формат входа `[TOPIC=t_i] text`, список тем, максимальная длина токенизации и соответствие классов `-`/`+` числовым id.


In [ ]:
direction_cfg_path = Path(hf_hub_download(
    repo_id=model_id,
    filename='direction_model_config.json',
    revision=model_revision,
    cache_dir=str(hf_cache_dir),
))
hf_config_path = Path(hf_hub_download(
    repo_id=model_id,
    filename='config.json',
    revision=model_revision,
    cache_dir=str(hf_cache_dir),
))
direction_cfg = json.loads(direction_cfg_path.read_text(encoding='utf-8'))
max_length = int(direction_cfg.get('max_length', 512))
topics = list(direction_cfg.get('topics') or DEFAULT_TOPICS)

with open(hf_config_path, 'r', encoding='utf-8') as f:
    hf_config = json.load(f)

id2label = {int(k): v for k, v in hf_config['id2label'].items()}
raw_label2id = hf_config.get('label2id', {})
if raw_label2id and all(str(v).lstrip('-').isdigit() for v in raw_label2id.values()):
    label2id = {str(k): int(v) for k, v in raw_label2id.items()}
else:
    label2id = {label: idx for idx, label in id2label.items()}
plus_id = label2id.get('+', 1)
minus_id = label2id.get('-', 0)

model_summary = {
    'model_name': direction_cfg.get('model_name'),
    'task': direction_cfg.get('task'),
    'input_format': direction_cfg.get('input_format'),
    'architecture': hf_config.get('architectures', []),
    'model_type': hf_config.get('model_type'),
    'hidden_size': hf_config.get('hidden_size'),
    'num_hidden_layers': hf_config.get('num_hidden_layers'),
    'vocab_size': hf_config.get('vocab_size'),
    'id2label': id2label,
    'max_length': max_length,
    'topics': topics,
}
print(json.dumps(model_summary, ensure_ascii=False, indent=2))


### Шаг 5. Загрузка `RELEVANCE.parquet` и опциональный фильтр `relevant = 0`

Загружается parquet из точного Kaggle-пути или локального fallback `data/RELEVANCE.parquet`. Если во входе есть колонка `relevant`, строки с `relevant = 0` исключаются; если такой колонки нет, фильтр пропускается и используются уже готовые topic-предсказания (`pred_t*`, `bert_topic_labels`).


In [ ]:
base = pd.read_parquet(input_path)
rows_before_relevant_filter = len(base)
if 'relevant' in base.columns:
    relevant_values = pd.to_numeric(base['relevant'], errors='coerce').fillna(0)
    base = base[relevant_values != 0].copy()
    print('rows before relevant filter:', rows_before_relevant_filter)
    print('excluded relevant=0 rows:', rows_before_relevant_filter - len(base))
    print('rows after relevant filter:', len(base))
else:
    print('column relevant not found; relevant=0 filter skipped')
rows_after_relevant_filter = len(base)

if MAX_ROWS is not None:
    base = base.head(int(MAX_ROWS)).copy()

if TEXT_COL not in base.columns:
    raise ValueError(f'Missing text column: {TEXT_COL}')
if ID_COL not in base.columns:
    base[ID_COL] = np.arange(len(base)).astype(str)

base[ID_COL] = base[ID_COL].astype(str)
base[TEXT_COL] = base[TEXT_COL].fillna('').astype(str)

print('base rows:', len(base))
print('base columns:', list(base.columns))
base.head(3)


### Шаг 6. Построение topic-post пар

Для оставшихся постов темы берутся из topic-предсказаний (`pred_t*_relevant`, `pred_t*` или `bert_topic_labels`). Для каждой выбранной темы создаётся строка `model_text = [TOPIC=t_i] text`, которую ожидает `final_direction_model`.


In [ ]:
def truthy(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {'1', 'true', 'yes', 'y', '+', 'relevant'}
    return bool(value)


def clean_direction(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value in {'+', '-'} else None


def topics_from_prediction_row(row, topics):
    selected = []
    for topic in topics:
        pred_col = f'pred_{topic}_relevant'
        short_pred_col = f'pred_{topic}'
        if pred_col in row.index and truthy(row[pred_col]):
            selected.append(topic)
        elif short_pred_col in row.index and truthy(row[short_pred_col]):
            selected.append(topic)
    if selected:
        return selected
    if 'bert_topic_labels' in row.index and isinstance(row['bert_topic_labels'], str) and row['bert_topic_labels'].strip():
        return [x.strip() for x in row['bert_topic_labels'].split(';') if x.strip() in topics]
    return []


def topics_from_gold_row(row, topics):
    selected = []
    for topic in topics:
        col = f'{topic}_relevant'
        if col in row.index and truthy(row[col]):
            selected.append(topic)
    return selected


def infer_topic_source(df, topic_pred_path, requested):
    requested = requested.lower()
    if requested not in {'auto', 'predicted', 'gold', 'all'}:
        raise ValueError("PAIR_TOPIC_SOURCE must be one of: auto, predicted, gold, all")

    has_predicted = any(
        f'pred_{topic}' in df.columns or f'pred_{topic}_relevant' in df.columns for topic in topics
    ) or 'bert_topic_labels' in df.columns
    has_gold = any(f'{topic}_relevant' in df.columns for topic in topics)

    if requested == 'predicted':
        if not has_predicted:
            raise ValueError('PAIR_TOPIC_SOURCE="predicted" requested, but no topic prediction columns were found.')
        return 'predicted'
    if requested == 'gold':
        if not has_gold:
            raise ValueError('PAIR_TOPIC_SOURCE="gold" requested, but no t*_relevant columns were found.')
        return 'gold'
    if requested == 'all':
        return 'all'
    if has_predicted:
        return 'predicted'
    if has_gold:
        return 'gold'
    if RUN_ALL_TOPICS_IF_NO_TOPIC_OUTPUT:
        return 'all'
    raise ValueError('No topic source found. Run topic inference first, provide t*_relevant columns, or enable all-topic fallback.')


# If topic notebook output exists, use it as the base table for predicted mode.
if topic_pred_path is not None and PAIR_TOPIC_SOURCE.lower() in {'auto', 'predicted'}:
    candidate_base = pd.read_parquet(topic_pred_path)
    if 'relevant' in candidate_base.columns:
        candidate_relevant = pd.to_numeric(candidate_base['relevant'], errors='coerce').fillna(0)
        candidate_base = candidate_base[candidate_relevant != 0].copy()
    if MAX_ROWS is not None:
        candidate_base = candidate_base.head(int(MAX_ROWS)).copy()
    if TEXT_COL in candidate_base.columns:
        if ID_COL not in candidate_base.columns:
            candidate_base[ID_COL] = np.arange(len(candidate_base)).astype(str)
        candidate_base[ID_COL] = candidate_base[ID_COL].astype(str)
        candidate_base[TEXT_COL] = candidate_base[TEXT_COL].fillna('').astype(str)
        base = candidate_base
    else:
        print(f'Topic prediction file ignored because it has no {TEXT_COL!r} column:', topic_pred_path)

topic_source = infer_topic_source(base, topic_pred_path, PAIR_TOPIC_SOURCE)
print('topic_source:', topic_source)

metadata_cols = [
    'channel_id', 'channel_name', 'message_id', 'post_date', 'source_file',
    'text_hash', 'prefilter_source', 'source', 'annotated', 'relevant',
]
records = []
for _, row in base.iterrows():
    if topic_source == 'predicted':
        row_topics = topics_from_prediction_row(row, topics)
    elif topic_source == 'gold':
        row_topics = topics_from_gold_row(row, topics)
    else:
        row_topics = topics

    for topic in row_topics:
        rec = {
            ID_COL: row[ID_COL],
            'topic': topic,
            TEXT_COL: row[TEXT_COL],
            'model_text': f'[TOPIC={topic}] {row[TEXT_COL]}',
            'topic_source': topic_source,
        }
        rel_col = f'{topic}_relevant'
        if rel_col in row.index:
            rec['topic_relevant'] = bool(truthy(row[rel_col]))
        direction = clean_direction(row.get('direction')) if 'direction' in row.index else None
        if direction is None:
            direction = clean_direction(row.get(f'{topic}_direction')) if f'{topic}_direction' in row.index else None
        if direction is not None:
            rec['direction'] = direction
        if 'direction_reasoning' in row.index:
            rec['direction_reasoning'] = row['direction_reasoning']
        for col in metadata_cols:
            if col in row.index:
                rec[col] = row[col]
        records.append(rec)

long_df = pd.DataFrame(records)
if LIMIT_PAIRS is not None:
    long_df = long_df.head(int(LIMIT_PAIRS)).copy()
if long_df.empty:
    raise ValueError('No topic-post pairs were built.')

print('direction pairs:', len(long_df))
print('unique posts:', long_df[ID_COL].nunique())
print(long_df['topic'].value_counts().sort_index())
long_df.head(3)


### Шаг 7. Загрузка tokenizer и final_direction_model

Tokenizer и модель загружаются через `from_pretrained()` из Hugging Face repo `wairado/post_direction_notator`. На GPU используется `float16`; на CPU остаётся обычная точность.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = BATCH_SIZE_GPU if device.type == 'cuda' else BATCH_SIZE_CPU

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=model_revision,
    cache_dir=str(hf_cache_dir),
    use_fast=True,
)
model_kwargs = {
    'revision': model_revision,
    'cache_dir': str(hf_cache_dir),
}
if device.type == 'cuda':
    model_kwargs['torch_dtype'] = torch.float16
model = AutoModelForSequenceClassification.from_pretrained(model_id, **model_kwargs)
model.to(device)
model.eval()

print('device:', device)
print('batch_size:', batch_size)
print('tokenizer:', tokenizer.__class__.__name__)
print('model:', model.__class__.__name__)


### Шаг 8. Инференс направления

Пары `[TOPIC=t_i] text` батчами прогоняются через direction-модель. Softmax даёт вероятности классов `-` и `+`, после чего выбирается класс с максимальной вероятностью.


In [ ]:
texts = long_df['model_text'].tolist()
all_probs = []
started = time.time()

for start in tqdm(range(0, len(texts), batch_size), desc='direction inference'):
    batch_texts = texts[start:start + batch_size]
    encoded = tokenizer(
        batch_texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors='pt',
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.inference_mode():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1).detach().float().cpu().numpy()
    all_probs.append(probs)

probs = np.vstack(all_probs) if all_probs else np.zeros((0, len(id2label)), dtype=np.float32)
pred_ids = probs.argmax(axis=1)
print('done in seconds:', round(time.time() - started, 2))
print('probs shape:', probs.shape)


### Шаг 9. Сохранение финального файла `preds.parquet`

Внутренний pair-level результат используется только для сборки итоговой широкой таблицы. На диск сохраняется один файл `preds.parquet`: одна строка = один пост, direction-предсказания разложены по колонкам `t1..t5`.


In [ ]:
result = long_df.copy()
result['prob_minus'] = probs[:, minus_id].astype('float32')
result['prob_plus'] = probs[:, plus_id].astype('float32')
result['direction_pred'] = [id2label[int(i)] for i in pred_ids]
result['direction_pred_id'] = pred_ids.astype('int8')
result['direction_margin_abs'] = np.abs(result['prob_plus'] - result['prob_minus']).astype('float32')

wide = base.copy()
wide[ID_COL] = wide[ID_COL].astype(str)
for topic in topics:
    topic_part = result[result['topic'] == topic].drop_duplicates(ID_COL).set_index(ID_COL)
    for col in ['direction_pred', 'direction_pred_id', 'prob_minus', 'prob_plus', 'direction_margin_abs']:
        wide[f'{topic}_{col}'] = wide[ID_COL].map(topic_part[col]) if len(topic_part) else np.nan

plus_map = result[result['direction_pred'] == '+'].groupby(ID_COL)['topic'].apply(lambda s: ';'.join(sorted(s)))
minus_map = result[result['direction_pred'] == '-'].groupby(ID_COL)['topic'].apply(lambda s: ';'.join(sorted(s)))
count_map = result.groupby(ID_COL)['topic'].size()
wide['direction_pair_count'] = wide[ID_COL].map(count_map).fillna(0).astype('int16')
wide['direction_plus_topics'] = wide[ID_COL].map(plus_map).fillna('')
wide['direction_minus_topics'] = wide[ID_COL].map(minus_map).fillna('')

out_parquet = WORKING / OUTPUT_FILE_NAME
wide.to_parquet(out_parquet, index=False, compression='zstd')

print('saved:', out_parquet)
print('rows saved:', len(wide))
wide.head()


### Шаг 10. Финальная статистика

В конце печатается статистика запуска: сколько постов и topic-post пар обработано, сколько строк попало в `preds.parquet`, распределение `+/-` по темам и сводка по фильтру `relevant = 0`. Дополнительные промежуточные файлы здесь не создаются.


In [ ]:
distribution = (
    result.groupby(['topic', 'direction_pred'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values(['topic', 'direction_pred'])
)
distribution['rate_within_topic'] = distribution['count'] / distribution.groupby('topic')['count'].transform('sum')

plus_by_topic = (
    result.assign(is_plus=(result['direction_pred'] == '+').astype(int))
    .groupby('topic')
    .agg(
        rows=('topic', 'size'),
        plus_count=('is_plus', 'sum'),
        plus_rate=('is_plus', 'mean'),
        avg_prob_plus=('prob_plus', 'mean'),
        avg_margin=('direction_margin_abs', 'mean'),
    )
    .reset_index()
    .sort_values('topic')
)

stats = {
    'input_path': str(input_path),
    'output_path': str(out_parquet),
    'hf_model_id': model_id,
    'hf_model_revision': model_revision,
    'topic_source': topic_source,
    'rows_input_before_relevant_filter': int(rows_before_relevant_filter),
    'rows_input_after_relevant_filter': int(rows_after_relevant_filter),
    'rows_excluded_relevant_0': int(rows_before_relevant_filter - rows_after_relevant_filter),
    'rows_saved_to_preds': int(len(wide)),
    'rows_direction_pairs': int(len(result)),
    'unique_posts_with_direction_pairs': int(result[ID_COL].nunique()),
    'posts_without_direction_pairs': int(len(wide) - result[ID_COL].nunique()),
    'topics': topics,
    'direction_counts': result['direction_pred'].value_counts().to_dict(),
    'topic_counts': result['topic'].value_counts().sort_index().to_dict(),
    'output_file_size_bytes': int(out_parquet.stat().st_size),
}

print(json.dumps(stats, ensure_ascii=False, indent=2))
display(distribution)
plus_by_topic


### Шаг 11. Очистка памяти и проверка итогового файла

Модель удаляется из памяти, CUDA cache очищается, затем печатается путь и размер единственного итогового файла `preds.parquet`.


In [ ]:
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print('Final prediction file:')
print(out_parquet, out_parquet.stat().st_size)
